In [9]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

poem = """One must have a mind of winter
To regard the frost and the boughs
Of the pine-trees crusted with snow;
And have been cold a long time
To behold the junipers shagged with ice,
The spruces rough in the distant glitter
Of the January sun; and not to think
Of any misery in the sound of the wind,
In the sound of a few leaves,
Which is the sound of the land
Full of the same wind
That is blowing in the same bare place
For the listener, who listens in the snow,
And, nothing himself, beholds
Nothing that is not there and the nothing that is."""

# Load GPT-2 tokenizer and model
model_name = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

def replace_last_word_with_seventh_highest_prob(poem, tokenizer, model):
    lines = poem.split('\n')
    modified_lines = []

    for line in lines:
        if not line.strip(): # Skip empty lines
            modified_lines.append(line)
            continue

        # Tokenize the line to identify the last word/token
        tokens = tokenizer.tokenize(line)
        if not tokens:
            modified_lines.append(line)
            continue

        # Find the index of the last actual word token
        last_word_idx = -1
        for i in range(len(tokens) - 1, -1, -1):
            if tokens[i].strip() and not tokens[i].startswith('Ġ'):
                last_word_idx = i
                break
            elif tokens[i].startswith('Ġ') and tokens[i][1:].strip():
                last_word_idx = i
                break

        if last_word_idx == -1: # No word found in the line
            modified_lines.append(line)
            continue

        # Get the token IDs for the part of the line BEFORE the last word
        # This handles the initial spacing correctly from the tokenizer's perspective
        pre_last_word_token_ids = tokenizer.encode(tokenizer.convert_tokens_to_string(tokens[:last_word_idx]))

        # Encode the input text to get predictions
        input_ids = torch.tensor([pre_last_word_token_ids])

        # Get model predictions
        with torch.no_grad():
            outputs = model(input_ids)
            predictions = outputs.logits

        # Get probabilities for the next token
        next_token_logits = predictions[0, -1, :]
        next_token_probs = torch.softmax(next_token_logits, dim=-1)

        # Get the sorted probabilities and their indices
        sorted_probs, sorted_indices = torch.sort(next_token_probs, descending=True)

        # Get the token with the 7th highest probability (0-indexed, so index 6)
        if len(sorted_indices) > 6:
            seventh_highest_token_id = sorted_indices[6].item()

            # Append the new token ID to the prefix token IDs
            new_line_token_ids = pre_last_word_token_ids + [seventh_highest_token_id]

            # Decode the entire sequence of token IDs back into a string
            # This ensures correct spacing as handled by the tokenizer
            new_line = tokenizer.decode(new_line_token_ids)
            modified_lines.append(new_line)
        else:
            # Not enough tokens to find the 7th highest, append original line
            modified_lines.append(line)

    return '\n'.join(modified_lines)

# Apply the function
modified_poem = replace_last_word_with_seventh_highest_prob(poem, tokenizer, model)
print("Original Poem:\n")
print(poem)
print("\n\nModified Poem:\n")
print(modified_poem)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Original Poem:

One must have a mind of winter
To regard the frost and the boughs
Of the pine-trees crusted with snow;
And have been cold a long time
To behold the junipers shagged with ice,
The spruces rough in the distant glitter
Of the January sun; and not to think
Of any misery in the sound of the wind,
In the sound of a few leaves,
Which is the sound of the land
Full of the same wind
That is blowing in the same bare place
For the listener, who listens in the snow,
And, nothing himself, beholds
Nothing that is not there and the nothing that is.


Modified Poem:

One must have a mind of her
To regard the frost and the bough in
Of the pine-trees crusted with snow that
And have been cold a long way
To behold the junipers shagged with ice as
The spruces rough in the distant horizon
Of the January sun; and not to have
Of any misery in the sound of the wind that
In the sound of a few leaves coming
Which is the sound of the voice
Full of the same day
That is blowing in the same bare air
F

In [3]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

poem = """One must have a mind of winter
To regard the frost and the boughs
Of the pine-trees crusted with snow;
And have been cold a long time
To behold the junipers shagged with ice,
The spruces rough in the distant glitter
Of the January sun; and not to think
Of any misery in the sound of the wind,
In the sound of a few leaves,
Which is the sound of the land
Full of the same wind
That is blowing in the same bare place
For the listener, who listens in the snow,
And, nothing himself, beholds
Nothing that is not there and the nothing that is."""

# Load GPT-2 tokenizer and model
model_name = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

def generate_xth_highest_prob_replacement(poem, tokenizer, model, x_value):
    lines = poem.split('\n')
    modified_lines = []

    for line in lines:
        if not line.strip(): # Skip empty lines
            modified_lines.append(line)
            continue

        tokens = tokenizer.tokenize(line)
        if not tokens:
            modified_lines.append(line)
            continue

        last_word_idx = -1
        for i in range(len(tokens) - 1, -1, -1):
            if tokens[i].strip() and not tokens[i].startswith('Ġ'):
                last_word_idx = i
                break
            elif tokens[i].startswith('Ġ') and tokens[i][1:].strip():
                last_word_idx = i
                break

        if last_word_idx == -1:
            modified_lines.append(line)
            continue

        # Get the token IDs for the part of the line BEFORE the last word
        pre_last_word_token_ids = tokenizer.encode(tokenizer.convert_tokens_to_string(tokens[:last_word_idx]))
        input_ids = torch.tensor([pre_last_word_token_ids])

        # Get model predictions
        with torch.no_grad():
            outputs = model(input_ids)
            predictions = outputs.logits

        # Get probabilities for the next token
        next_token_logits = predictions[0, -1, :]
        next_token_probs = torch.softmax(next_token_logits, dim=-1)

        # Get the sorted probabilities and their indices
        sorted_probs, sorted_indices = torch.sort(next_token_probs, descending=True)

        # Get the token with the x-th highest probability (0-indexed, so x_value - 1)
        if len(sorted_indices) >= x_value:
            xth_highest_token_id = sorted_indices[x_value - 1].item()

            # Append the new token ID to the prefix token IDs
            new_line_token_ids = pre_last_word_token_ids + [xth_highest_token_id]

            # Decode the entire sequence of token IDs back into a string
            new_line = tokenizer.decode(new_line_token_ids)
            modified_lines.append(new_line)
        else:
            # If not enough tokens for the x-th highest, append original line
            modified_lines.append(line)

    return '\n'.join(modified_lines)

# Choose an x_value for the replacement
# Experiment with different values to find funny, witty, or absurd results.
x_value_for_replacement = 45 # Example: 1st, 2nd, 7th, 10th, 20th highest probability

# Apply the function
modified_poem = generate_xth_highest_prob_replacement(poem, tokenizer, model, x_value=x_value_for_replacement)

print("Original Poem:\n")
print(poem)
print(f"\n\nModified Poem (x_value={x_value_for_replacement}):\n")
print(modified_poem)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Original Poem:

One must have a mind of winter
To regard the frost and the boughs
Of the pine-trees crusted with snow;
And have been cold a long time
To behold the junipers shagged with ice,
The spruces rough in the distant glitter
Of the January sun; and not to think
Of any misery in the sound of the wind,
In the sound of a few leaves,
Which is the sound of the land
Full of the same wind
That is blowing in the same bare place
For the listener, who listens in the snow,
And, nothing himself, beholds
Nothing that is not there and the nothing that is.


Modified Poem (x_value=45):

One must have a mind of awe
To regard the frost and the bough when
Of the pine-trees crusted with snowgl
And have been cold a long fucking
To behold the junipers shagged with ice cold
The spruces rough in the distant darkness
Of the January sun; and not to ask
Of any misery in the sound of the wind can
In the sound of a few leaves emerging
Which is the sound of the storm
Full of the same spirit
That is blowing 